# ML-07 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedosrf/flyrank-ml-internship-ahmedosrf/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

I keep Lane 2: **content-refresh opportunity scoring**. This notebook audits two findings from the FlyRank April 2026 report and then attacks my Week-5 model before treating its numbers as decision support. The report source is the public [State of AI-Driven SEO](https://state-of-seo-2026.flyrank.ai/).

## 1. Two paper findings + my methodology questions

### Finding A — The Content Results Curve

The report observes that average Health Score is highest for pages aged 61–90 days (**37.2, n=18.2K**) and lower for pages aged 271–365 days (**29.6, n=21.0K**); the 365+ group is **35.5, n=21.4K**. I read this as an **observed age-bucket association**, not proof that age itself causes the score change.

My constructive methodology question is: **How are page outcomes and age buckets linked over time, and how much repeated client/page mix is present in each bucket?** A grouped or time-aware design, or an adjustment for client mix and publication cohort, would help separate an age association from differences in sites, topics, and measurement windows. I would also ask whether the result is stable across clients rather than being driven by a few large portfolios.

### Finding B — Click Capture by Position Tier

The report observes weighted CTR of **0.420%** for Top 3, **0.340%** for positions 4–10, **0.163%** for positions 21–50, and **0.050%** for Deep pages. It recommends prioritizing pages that already rank visibly for snippet refinement. I read this as a **directional cross-sectional pattern**, not a causal estimate of what a snippet edit will produce.

My constructive methodology question is: **Are these pooled weighted CTR values stable across client, query intent, device, and time, and what is the effective sample size in each position tier?** Position and intent can be confounded: pages in different tiers may serve different queries and audiences. A grouped/time-aware check and uncertainty intervals would strengthen the action recommendation, while the disclosed lack of p-values and confidence intervals is an important limitation.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

REPO = Path('/home/ubuntu/flyrank-ml-internship-starter')
DATA = REPO / 'data/raw/content_refresh_anonymized.csv'
OUT_DIR = REPO / 'work/outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

df = pd.read_csv(DATA)
df['target'] = (df['trend_direction'] == 'down').astype(int)
NUMERIC_FEATURES = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position',
    'engagement_rate', 'content_age_days', 'days_since_last_update',
    'word_count', 'search_volume', 'cpc'
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN = {'trend_direction', 'trend_pct', 'is_declining_label', 'target', 'leaky_target_copy'}
assert not (set(MODEL_FEATURES) & FORBIDDEN)
assert set(MODEL_FEATURES).issubset(df.columns)
print(f'Rows: {len(df):,} | clients: {df.client_id.nunique()} | target rate: {df.target.mean():.4f}')
print('Clean feature count:', len(MODEL_FEATURES), '| forbidden overlap:', set(MODEL_FEATURES) & FORBIDDEN)

Rows: 30,000 | clients: 32 | target rate: 0.5421
Clean feature count: 19 | forbidden overlap: set()


## 2. My model under an honest split (before/after)

The Week-5 model is Logistic Regression with imputation, scaling, and one-hot encoding. The **before** number below is a random row split, which can place rows from the same client in both sets. The **after** number is a grouped holdout by `client_id`, which asks the more deployment-relevant question: does the ranking transfer to clients not seen during training? Both runs use the same features, seed, target, and Precision@K implementation. The target is an evaluation proxy derived from `trend_direction`; it is never a model input.

In [2]:
def make_model():
    preprocess = ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), NUMERIC_FEATURES),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ('preprocess', preprocess),
        ('classifier', LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED))
    ])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind='mergesort')[:k]
    return float(np.asarray(labels)[order].mean())

def evaluate_split(train, test, split_name):
    model = make_model()
    model.fit(train[MODEL_FEATURES], train.target)
    scores = model.predict_proba(test[MODEL_FEATURES])[:, 1]
    result = {
        'split': split_name,
        'train_rows': int(len(train)),
        'test_rows': int(len(test)),
        'train_clients': int(train.client_id.nunique()),
        'test_clients': int(test.client_id.nunique()),
        'precision_at_10': precision_at_k(scores, test.target, 10),
        'precision_at_50': precision_at_k(scores, test.target, 50),
        'average_precision': average_precision_score(test.target, scores),
        'roc_auc': roc_auc_score(test.target, scores),
        'test_base_rate': float(test.target.mean()),
    }
    return model, scores, result

X = df[MODEL_FEATURES]
y = df.target
groups = df.client_id
row_train_idx, row_test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=SEED, stratify=y)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
group_train_idx, group_test_idx = next(gss.split(X, y, groups=groups))
row_train, row_test = df.iloc[row_train_idx].copy(), df.iloc[row_test_idx].copy()
group_train, group_test = df.iloc[group_train_idx].copy(), df.iloc[group_test_idx].copy()

row_model, row_scores, row_result = evaluate_split(row_train, row_test, 'random row split (before)')
group_model, group_scores, group_result = evaluate_split(group_train, group_test, 'grouped client holdout (after)')
assert set(group_train.client_id).isdisjoint(set(group_test.client_id))
comparison = pd.DataFrame([row_result, group_result])
display(comparison.round(4))
print('Client overlap after grouped split:', len(set(group_train.client_id) & set(group_test.client_id)))
print('Grouped split is the primary honest estimate; the random split is shown only to expose the validation-design gap.')

,split,train_rows,test_rows,train_clients,test_clients,precision_at_10,precision_at_50,average_precision,roc_auc,test_base_rate
0,random row split (before),24000,6000,32,31,0.9,0.86,0.6826,0.6684,0.542
1,grouped client holdout (after),23837,6163,25,7,0.8,0.58,0.5386,0.5503,0.511


Client overlap after grouped split: 0
Grouped split is the primary honest estimate; the random split is shown only to expose the validation-design gap.


The grouped result is the one I would use for a cautious deployment discussion. A gap between random and grouped scores is evidence that row-level validation may benefit from client-specific similarity. Even if the grouped score remains above the base rate, it is a measured ranking result on these held-out clients, not evidence that an edit will cause recovery.

## 3. Leakage audit

The clean feature list excludes the label, its siblings, future/trend fields, and existing decision flags. I also run a deliberate diagnostic: adding a direct copy of the target should make the score nearly perfect. That jump is a useful test of the harness, but the column is immediately removed and is not part of the reported model.

In [3]:
label_related = {'trend_direction', 'trend_pct', 'is_declining_label', 'target'}
flag_related = {'optimization_flags', 'baseline_refresh_score', 'baseline_action', 'reason_code'}
print('Label/future/decision fields present in clean features:', sorted((set(MODEL_FEATURES) & (label_related | flag_related))))
assert not (set(MODEL_FEATURES) & (label_related | flag_related))

# Diagnostic only: a direct target copy should be detected and produce a perfect ranking.
probe = group_train.copy()
probe_test = group_test.copy()
probe['leaky_target_copy'] = probe.target
probe_test['leaky_target_copy'] = probe_test.target
leaky_features = ['leaky_target_copy']
leaky_model = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED))
])
leaky_model.fit(probe[leaky_features], probe.target)
leaky_scores = leaky_model.predict_proba(probe_test[leaky_features])[:, 1]
leaky_ap = average_precision_score(probe_test.target, leaky_scores)
leaky_auc = roc_auc_score(probe_test.target, leaky_scores)
print(f'Deliberate leakage probe AP={leaky_ap:.4f}, ROC AUC={leaky_auc:.4f} (diagnostic only)')
assert 'leaky_target_copy' not in MODEL_FEATURES
print('Final clean feature audit: PASS — no label-derived, future-window, or product-flag input is used.')

Label/future/decision fields present in clean features: []
Deliberate leakage probe AP=1.0000, ROC AUC=1.0000 (diagnostic only)
Final clean feature audit: PASS — no label-derived, future-window, or product-flag input is used.


## 4. Claim rewrite

**Earlier claim:** “The Logistic Regression model beats the baseline and can identify pages that will recover after editing.”

**Safer rewrite:** “On this starter snapshot and fixed grouped client holdout, the Logistic Regression ranking had higher measured Precision@10 and Precision@50 than the Week-4 rule in the preceding experiment. The result is directional decision-support for choosing pages to review; it does not establish that the model causes recovery, and it may not transfer to new time periods or client portfolios without a time-aware and prospective check.”

**What the errors say:** the model still confuses high-exposure pages with low recent momentum and misses some declining pages with weak volume or mixed engagement signals. Those errors make human review and follow-up measurement necessary.

In [4]:
# Error examples use anonymized row numbers and public-safe aggregates rather than client names or private queries.
error_view = group_test[['target'] + NUMERIC_FEATURES].copy()
error_view['model_score'] = group_scores
error_view['predicted_at_0_5'] = (group_scores >= 0.5).astype(int)
error_view['error_type'] = np.where(
    (error_view.predicted_at_0_5 == 1) & (error_view.target == 0), 'false_positive',
    np.where((error_view.predicted_at_0_5 == 0) & (error_view.target == 1), 'false_negative', 'correct')
)
errors = error_view[error_view.error_type != 'correct'].copy()
print(f'Grouped threshold errors: {len(errors):,} | false positives: {(errors.error_type == "false_positive").sum():,} | false negatives: {(errors.error_type == "false_negative").sum():,}')
error_examples = errors.head(3)[['error_type','target','model_score','impressions_90d','ctr','avg_position','content_age_days']].reset_index(drop=True)
display(error_examples.round(4))

metrics = {
    'seed': SEED,
    'paper_source': 'https://state-of-seo-2026.flyrank.ai/',
    'before_random_split': row_result,
    'after_grouped_client_split': group_result,
    'leakage_probe': {'average_precision': float(leaky_ap), 'roc_auc': float(leaky_auc), 'used_in_final_model': False},
    'clean_features': MODEL_FEATURES,
    'error_counts': {
        'threshold_errors': int(len(errors)),
        'false_positives': int((errors.error_type == 'false_positive').sum()),
        'false_negatives': int((errors.error_type == 'false_negative').sum()),
    },
    'claims': 'observed/measured/directional/decision-support only; no causal claim',
}
(OUT_DIR / 'ml07_validation_audit_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
print('Wrote:', OUT_DIR / 'ml07_validation_audit_metrics.json')

Grouped threshold errors: 2,789 | false positives: 1,617 | false negatives: 1,172


,error_type,target,model_score,impressions_90d,ctr,avg_position,content_age_days
0,false_positive,0,0.7434,307,0.00,39.8,238
1,false_negative,1,0.4635,99,2.02,6.9,187
2,false_negative,1,0.3920,297,0.34,13.9,502


Wrote: /home/ubuntu/flyrank-ml-internship-starter/work/outputs/ml07_validation_audit_metrics.json


## Self-check

- [x] Two paper findings are named with constructive methodology questions.
- [x] The model is rerun on a random row split and an honest grouped client holdout.
- [x] The same target, features, seed, and ranking metrics are used for the before/after comparison.
- [x] The clean feature set excludes label-derived, future-window, and product-decision fields.
- [x] A deliberate leakage probe is shown and explicitly excluded from the final model.
- [x] Real grouped-holdout error examples are shown with public-safe aggregates.
- [x] Claims use observed, measured, directional, and decision-support language rather than causal wording.
- [x] The notebook runs top to bottom and writes a metrics receipt under `work/outputs/`.
